# 01 Bronze profiling

Download (or reuse) the verbatim source CSV, profile it, and confirm the
facts recorded during planning. Nothing is cleaned here.

Run this notebook from PyCharm using the IDE **Managed Jupyter server**:
`Project Settings > Python Interpreter > Jupyter`, let PyCharm start and
manage the server, and select the project `.venv` as the kernel. No separate
`jupyter notebook` process is required (see `notebooks/README.md`).

## Bootstrap

Insert the repo root on `sys.path` and pin SQLite.

In [8]:
import os
import sys
from pathlib import Path

# Make the project importable no matter where the kernel starts.
_here = Path.cwd()
_root = _here.parent if _here.name == "notebooks" else _here
sys.path.insert(0, str(_root))
os.chdir(_root)

# Force SQLite so notebooks never touch PostgreSQL by accident.
os.environ["ENV"] = "DEV"
os.environ.setdefault("ETL_SQLITE_PATH", "data/warehouse/violence_staging.db")

import pandas as pd  # noqa: F401  (used by later cells)

from violence_etl.config.settings import get_settings

settings = get_settings()
print("env    :", settings.env)
print("sqlite :", settings.sqlite_path)
print("bronze :", settings.bronze_dir)

env    : DEV
sqlite : data/warehouse/violence_staging.db
bronze : data/bronze


## Download and land the source file

The landing file is named by the sha256 of its bytes.

In [9]:

from violence_etl.utils import io as etl_io

data = etl_io.download_source(settings.source_url)
path, sha, written = etl_io.write_landing(settings.bronze_dir, data)
print("bytes :", len(data))
print("sha256:", sha)
print("landing:", path)
print("written:", written)

bytes : 993379
sha256: f8d452b45dd5fc47426862f25d98129f0bf1cc589b5aa50bda041955e2ea4ea3
landing: data/bronze/landing/f8d452b45dd5fc47426862f25d98129f0bf1cc589b5aa50bda041955e2ea4ea3.csv
written: False


## Profile the raw CSV

All fields stay text (`dtype=str`, `keep_default_na=False`).

In [10]:
import pandas as pd

df = etl_io.read_csv(path)
print("shape:", df.shape)
print("columns:", list(df.columns))
df.dtypes

shape: (15981, 7)
columns: ['FECHA HECHO', 'COD_DEPTO', 'DEPARTAMENTO', 'COD_MUNI', 'MUNICIPIO', 'ZONA', 'CANTIDAD']


FECHA HECHO     object
COD_DEPTO       object
DEPARTAMENTO    object
COD_MUNI        object
MUNICIPIO       object
ZONA            object
CANTIDAD        object
dtype: object

In [11]:
# Empty strings are preserved, not coerced to NaN.
empty_counts = (df == "").sum()
print("empty cells per column:")
print(empty_counts)

empty cells per column:
FECHA HECHO     0
COD_DEPTO       0
DEPARTAMENTO    0
COD_MUNI        0
MUNICIPIO       0
ZONA            0
CANTIDAD        0
dtype: int64


In [12]:
# Expected planning facts: ZONA in {URBANA, RURAL}, CANTIDAD always 1,
# 32 departments, 777 municipality codes.
print(df["ZONA"].value_counts())
print("CANTIDAD unique:", df["CANTIDAD"].unique())
print("distinct departments:", df["COD_DEPTO"].nunique())
print("distinct municipalities:", df["COD_MUNI"].nunique())

ZONA
URBANA    9993
RURAL     5988
Name: count, dtype: int64
CANTIDAD unique: ['1']
distinct departments: 32
distinct municipalities: 779


In [13]:
# Duplicate groups and date range (dates are DD/MM/YYYY, parsed explicitly).
fechas = pd.to_datetime(df["FECHA HECHO"], format="%d/%m/%Y", errors="coerce")
print("date min/max:", fechas.min(), fechas.max())
print("distinct full rows:", df.drop_duplicates().shape[0])
print("exact duplicate rows:", df.shape[0] - df.drop_duplicates().shape[0])
business = df.assign(fecha=fechas).drop_duplicates(
    ["fecha", "COD_DEPTO", "COD_MUNI", "ZONA", "CANTIDAD"]
)
print("distinct business rows:", business.shape[0])

date min/max: 2003-01-01 00:00:00 2026-10-31 00:00:00
distinct full rows: 14460
exact duplicate rows: 1521
distinct business rows: 14460


## Ingest into bronze and inspect the mirrored table

In [14]:
from violence_etl.transformations.bronze import ingest_bronze
from violence_etl.db.connection import make_engine

engine = make_engine(settings=settings)
result = ingest_bronze(engine, settings, csv_path=path)
result

{'status': 'loaded',
 'source_sha256': 'f8d452b45dd5fc47426862f25d98129f0bf1cc589b5aa50bda041955e2ea4ea3',
 'source_file': 'data/bronze/raw/yi5j-5fe9_20261118T213601Z.csv',
 'row_count': 15981,
 'raw_rows': 15981,
 'elapsed_seconds': 0.21}

In [15]:
pd.read_sql("select * from bronze_crimes_raw order by row_num limit 5", engine)

,raw_id,source_sha256,source_file,source_url,load_ts,row_num,fecha_hecho_raw,cod_depto_raw,departamento_raw,cod_muni_raw,municipio_raw,zona_raw,cantidad_raw
0,1,f8d452b45dd5fc47426862f25d98129f0bf1cc589b5aa5...,data/bronze/raw/yi5j-5fe9_20261118T213601Z.csv,http://www.datos.gov.co/api/v3/views/yi5j-5fe9...,2026-11-18 21:36:01.706273,1,31/10/2026,19,CAUCA,19392,LA SIERRA,RURAL,1
1,2,f8d452b45dd5fc47426862f25d98129f0bf1cc589b5aa5...,data/bronze/raw/yi5j-5fe9_20261118T213601Z.csv,http://www.datos.gov.co/api/v3/views/yi5j-5fe9...,2026-11-18 21:36:01.706273,2,31/10/2026,41,HUILA,41020,ALGECIRAS,URBANA,1
2,3,f8d452b45dd5fc47426862f25d98129f0bf1cc589b5aa5...,data/bronze/raw/yi5j-5fe9_20261118T213601Z.csv,http://www.datos.gov.co/api/v3/views/yi5j-5fe9...,2026-11-18 21:36:01.706273,3,31/10/2026,19,CAUCA,19001,POPAYAN,URBANA,1
3,4,f8d452b45dd5fc47426862f25d98129f0bf1cc589b5aa5...,data/bronze/raw/yi5j-5fe9_20261118T213601Z.csv,http://www.datos.gov.co/api/v3/views/yi5j-5fe9...,2026-11-18 21:36:01.706273,4,30/10/2026,81,ARAUCA,81001,ARAUCA,URBANA,1
4,5,f8d452b45dd5fc47426862f25d98129f0bf1cc589b5aa5...,data/bronze/raw/yi5j-5fe9_20261118T213601Z.csv,http://www.datos.gov.co/api/v3/views/yi5j-5fe9...,2026-11-18 21:36:01.706273,5,30/10/2026,81,ARAUCA,81736,SARAVENA,URBANA,1


In [16]:
pd.read_sql(
    "select count(*) as raw_rows, count(distinct source_sha256) as shas "
    "from bronze_crimes_raw",
    engine,
)

,raw_rows,shas
0,15981,1


The bronze table mirrors the source verbatim: 15,696 rows for the real
dataset, every source field stored as text, plus `source_url`, `source_file`,
`source_sha256`, `load_ts` and 1-based `row_num`.